Processing DMS files from preprocessed_substitutions

In [1]:
import os
import shutil
import sys

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_rows", 20, "display.max_columns", None)

sys.path.append("../scripts/")
from dms_preprocessing_utils import check_wt, hgvs_to_1aa, mut_pos_regex, count_missense_non_na

In [2]:
# Inputs:
reference_file = "/n/groups/marks/projects/indels_human/anon/reference_files/DMS_substitutions.csv"
# reference_file = "/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/reference_files/DMS_substitutions_WIP_2023_10_18.csv"
dms_preprocessed_folder = "/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/DMS_assays/preprocessed_substitutions/"
# dms_preprocessed_folder_nathan = "/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/DMS_assays/raw_data/Nathan_2023_staging/assays_offsets_fixed/"

# Output:
dms_processed_folder_out = "/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/DMS_assays/substitutions"

In [3]:
ref_df = pd.read_csv(reference_file)
# When using the reference file with "excluded" version, we need to use strings for ProteinGym_version
keep_versions = [0.1, 1]
ref_df = ref_df[ref_df["ProteinGym_version"].isin(keep_versions)]
ref_df["ProteinGym_version"] = ref_df["ProteinGym_version"].astype(float)
assert len(ref_df) > 0 

In [4]:
# Quick check: missing files
dms_folder = "/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/DMS_assays/substitutions"
dms_files = os.listdir(dms_folder)
ref_df[~ref_df["DMS_filename"].isin(dms_files)]

,DMS_id,DMS_filename,UniProt_ID,taxon,source_organism,target_seq,seq_len,includes_multiple_mutants,DMS_total_number_mutants,DMS_number_single_mutants,DMS_number_multiple_mutants,DMS_binarization_cutoff,DMS_binarization_method,first_author,title,year,jo,region_mutated,molecule_name,selection_assay,selection_type,library_construction,replicates,MSA_filename,MSA_start,MSA_end,MSA_len,MSA_bitscore,MSA_theta,MSA_num_seqs,MSA_perc_cov,MSA_num_cov,MSA_N_eff,MSA_Neff_L,MSA_Neff_L_category,MSA_num_significant,MSA_num_significant_L,raw_DMS_filename,raw_DMS_phenotype_name,raw_DMS_directionality,raw_DMS_mutant_column,weight_file_name,reviewed_by,Comments,Old_DMS_ID,Old_UniProt_ID,Old_target_seq,MSA_comments,ProteinGym_version,Reason excluded,raw_mut_offset


In [14]:
# TODO import proteingym.DMS_processing_utils (copy from tranception repo?)
# Run on each of the DMS files
AA_vocab = "ACDEFGHIKLMNPQRSTVWY"

def get_mutated_sequence(focus_seq, mutant, start_idx=1, AA_vocab=AA_vocab):
    """
    Helper function that mutates an input sequence (focus_seq) via an input mutation triplet (substitutions only).
    Mutation triplet are typically based on 1-indexing: start_idx is used for switching to 0-indexing.
    """
    mutated_seq = list(focus_seq)
    for mutation in mutant.split(":"):
        try:
            from_AA, position, to_AA = mutation[0], int(mutation[1:-1]), mutation[-1]
        except:
            print("Issue with mutant: "+str(mutation))
        relative_position = position - start_idx
        assert (from_AA==focus_seq[relative_position]), "Invalid from_AA or mutant position: "+str(mutation)+" from_AA: "+str(from_AA) + " relative pos: "+str(relative_position) + " focus_seq: "+str(focus_seq)
        assert (to_AA in AA_vocab) , "Mutant to_AA is invalid: "+str(mutation)
        mutated_seq[relative_position] = to_AA
    return "".join(mutated_seq)

def DMS_file_cleanup(DMS_filename, target_seq, DMS_binarization_threshold, start_idx=1, end_idx=None, DMS_mutant_column='mutant', DMS_phenotype_name='score', DMS_directionality=1, AA_vocab = "ACDEFGHIKLMNPQRSTVWY", debug=False, strict_wt_check=False):
    """
    Function to process the raw substitution DMS assay data (eg., removing invalid mutants, aggregate silent mutations).
    """
    DMS_data = pd.read_csv(DMS_filename, low_memory=False)
    assert DMS_phenotype_name in DMS_data.columns, f"Phenotype column not found: {DMS_phenotype_name}; columns: {DMS_data.columns}"
    end_idx = start_idx + len(target_seq) - 1 if end_idx is None else end_idx
    DMS_data['mutant'] = DMS_data[DMS_mutant_column]
    
    DMS_data=DMS_data[DMS_data['mutant'].notnull()].copy()
    DMS_data = DMS_data[~(DMS_data['mutant'] == "WT")]
    len_before = len(DMS_data)
    
    # Checking the following conditions but making a column for each:
    # DMS_data=DMS_data[DMS_data['mutant'].apply(lambda x: all([len(y)>=3 for y in x.split(":")]))].copy() # Mutant triplets should have at least 3 or more characters
    # DMS_data=DMS_data[DMS_data['mutant'].apply(lambda x: all([y[0]==target_seq[int(y[1:-1])-start_idx] for y in x.split(":")]))].copy()  # Checking all WT amino acids match target sequence
    # DMS_data=DMS_data[DMS_data['mutant'].apply(lambda x: all([(y[0] in AA_vocab) and (y[1:-1].isnumeric()) and (y[-1] in AA_vocab) for y in x.split(":")]))].copy()
    # DMS_data=DMS_data[DMS_data['mutant'].apply(lambda x: all([int(y[1:-1])-start_idx >=0 and int(y[1:-1]) <= end_idx for y in x.split(":")]))].copy()
    DMS_data["mutant_format"] = DMS_data['mutant'].apply(lambda x: all([len(y)>=3 for y in x.split(":")])) # Mutant triplets should have at least 3 or more characters
    DMS_data["mutant_valid_aa"] = DMS_data['mutant'].apply(lambda x: all([(y[0] in AA_vocab) and (y[1:-1].isnumeric()) and (y[-1] in AA_vocab) for y in x.split(":")]))
    DMS_data["match_wt"] = False
    
    DMS_data.loc[DMS_data["mutant_format"] & DMS_data["mutant_valid_aa"], "pos_max"] = DMS_data.loc[DMS_data["mutant_format"] & DMS_data["mutant_valid_aa"], "mutant"].apply(lambda x: max([int(y[1:-1]) for y in x.split(":")]))
    # assert pos_max <= len(target_seq), f"Maximum mutant position ({pos_max}) is greater than the length of the target sequence ({len(target_seq)})."
    if (DMS_data["pos_max"] > len(target_seq)).any():
        print(f"Warning: Some mutations have positions that are greater than target sequence length. Max mutation position: {pos_max.max()}, target seq length: {len(target_seq)}")
        print("Filtering mutations beyond target seq length") 
    DMS_data["valid_pos"] = (DMS_data["pos_max"] <= len(target_seq))
    
    # Dropping WT from dataframe 
    DMS_data.loc[DMS_data["mutant_format"] & DMS_data["mutant_valid_aa"] & DMS_data["valid_pos"], "match_wt"] = DMS_data.loc[DMS_data["mutant_format"] & DMS_data["mutant_valid_aa"] & DMS_data["valid_pos"], "mutant"].apply(lambda x: all([y[0]==target_seq[int(y[1:-1])-start_idx] for y in x.split(":")]))
    # Add synonymous filtering
    DMS_data["synonymous"] = DMS_data["mutant"].apply(lambda x: all([y[0]==y[-1] for y in x.split(":")]))
    
    num_wt_mismatch = (DMS_data["mutant_format"] & DMS_data["mutant_valid_aa"] & DMS_data["valid_pos"] & ~DMS_data["match_wt"] & ~DMS_data["synonymous"]).sum()

    if debug:
        print(f"Number of dropped mutant formats: {(~DMS_data['mutant_format']).sum()} / {len(DMS_data)} = {(~DMS_data['mutant_format']).sum()/len(DMS_data):.1%}")
        print(f"Number of dropped invalid amino acids: {(~DMS_data['mutant_valid_aa']).sum()} / {len(DMS_data)} = {(~DMS_data['mutant_valid_aa']).sum()/len(DMS_data):.1%}")
        print(f"Number of dropped synonymous mutants: {(DMS_data['synonymous']).sum()} / {len(DMS_data)} = {(DMS_data['synonymous']).sum()/len(DMS_data):.1%}")
        print(f"Number of dropped mutants not matching WT: {num_wt_mismatch} / {len(DMS_data)} = {num_wt_mismatch/len(DMS_data):.1%}")
        # If more than 5 WT check mutants are dropped, print warning
        if num_wt_mismatch > 5:
            print(f"Warning: More than 5 mutants ({num_wt_mismatch}) don't match WT target sequence. Do the positions index correctly into the target sequence?")
    
    if strict_wt_check:
        assert num_wt_mismatch == 0, f"Some {num_wt_mismatch} mutants don't match WT target sequence. Do the positions index correctly into the target sequence?. Examples: {DMS_data[(DMS_data['mutant_format'] & DMS_data['mutant_valid_aa'] & ~DMS_data['match_wt'])]['mutant'].head()}"
    
    DMS_data = DMS_data[DMS_data["mutant_format"] & DMS_data["mutant_valid_aa"] & DMS_data["match_wt"] & ~DMS_data["synonymous"] & DMS_data["valid_pos"]].copy()
    if len(DMS_data) != len_before:
        print(f"Dropped mutants: Length before: {len_before}, length after: {len(DMS_data)}: {1 - (len(DMS_data) / len_before):.1%} dropped")
        if 1 - (len(DMS_data) / len_before) > 0.1:
            print(f"Warning: Dropped more than 10% of mutants. This may indicate a problem with the DMS data.")
        
    assert len(DMS_data) > 0, f"All mutants have been filtered out, 0 length: {DMS_filename}"  # This might not always pick up...
    
    # Filter out synonymous mutants in multiples
    def remove_synon_multiple(muts):
        if ":" not in muts:
            return muts
        muts_split = muts.split(":")
        muts_missense = [m for m in muts_split if m[0] != m[-1]]
        # Don't filter out all, leave at least one, otherwise "mutant" is empty and will throw an error
        if len(muts_missense) == 0:
            muts_missense = muts_split[0]
        return ":".join(muts_missense)

    DMS_data["mutant"] = DMS_data["mutant"].apply(remove_synon_multiple)
    
    DMS_data[DMS_phenotype_name]=pd.to_numeric(DMS_data[DMS_phenotype_name],errors='coerce')
    DMS_data=DMS_data[np.isfinite(DMS_data[DMS_phenotype_name])]
    DMS_data.dropna(subset = [DMS_phenotype_name], inplace=True)
    DMS_directionality = float(DMS_directionality)
    if pd.isna(DMS_directionality):
        # TODO should this be a failure if not there?
        print("No DMS directionality found, using 1 by default.")
        DMS_directionality = 1
    DMS_data['DMS_score'] = DMS_data[DMS_phenotype_name] * DMS_directionality
    DMS_data=DMS_data[['mutant','DMS_score']]
    
    num_duplicate_mutants = DMS_data['mutant'].duplicated().sum()
    print(f"Debug: Number of duplicate mutants: {num_duplicate_mutants} ({num_duplicate_mutants / len(DMS_data):.2%})")
    
    # Average out mutations
    DMS_data=DMS_data.groupby('mutant').mean().reset_index()
    
    # Add DMS_binarization data
    if pd.isna(DMS_binarization_threshold):
        print("No binarization threshold given, will outputs NaNs in that column for now.")
        DMS_data["DMS_score_bin"] = np.nan
    else:
        DMS_data["DMS_score_bin"] = DMS_data["DMS_score"].apply(lambda x: 1 if x > (DMS_directionality * DMS_binarization_threshold) else 0)

    
    DMS_data['mutated_sequence'] = DMS_data['mutant'].apply(lambda x: get_mutated_sequence(target_seq, x))
    DMS_data=DMS_data[['mutant','mutated_sequence','DMS_score', "DMS_score_bin"]]
    print("Final length:", len(DMS_data))
    
    return DMS_data

## Note: Check up on this missing information in the file

In [15]:
# Some are missing raw DMS mutant column, will need to add in in original reference file processing notebook
necessary_cols = ["target_seq", "raw_DMS_phenotype_name","raw_DMS_directionality", "raw_DMS_mutant_column", "raw_DMS_directionality", "DMS_binarization_cutoff"]
nan_rows = ref_df[ref_df[necessary_cols].isna().any(axis=1)]
display(nan_rows)
nan_rows["reviewed_by"].value_counts()
assert len(nan_rows) == 0

,DMS_id,DMS_filename,UniProt_ID,taxon,source_organism,target_seq,seq_len,includes_multiple_mutants,DMS_total_number_mutants,DMS_number_single_mutants,DMS_number_multiple_mutants,DMS_binarization_cutoff,DMS_binarization_method,first_author,title,year,jo,region_mutated,molecule_name,selection_assay,selection_type,library_construction,replicates,MSA_filename,MSA_start,MSA_end,MSA_len,MSA_bitscore,MSA_theta,MSA_num_seqs,MSA_perc_cov,MSA_num_cov,MSA_N_eff,MSA_Neff_L,MSA_Neff_L_category,MSA_num_significant,MSA_num_significant_L,raw_DMS_filename,raw_DMS_phenotype_name,raw_DMS_directionality,raw_DMS_mutant_column,weight_file_name,reviewed_by,Comments,Old_DMS_ID,Old_UniProt_ID,Old_target_seq,MSA_comments,ProteinGym_version,Reason excluded,raw_mut_offset


In [16]:
import numpy as np
def flatten(l):
    return [item for sublist in l for item in sublist if sublist != np.NaN]

# Show an example here
dms_filename = "AMFR_HUMAN_Rocklin_2023_4G3O.csv"
orig_file = "AMFR_HUMAN_Rocklin_2023_4G3O.csv"
row = ref_df[ref_df["DMS_filename"] == dms_filename].iloc[0]
filename = dms_preprocessed_folder + "/" + orig_file
dms_bin_threshold = row["DMS_binarization_cutoff"]
tmp_df = pd.read_csv(filename)
target_seq = row["target_seq"]
# tmp_df["mutant_valid_aa"] = tmp_df['mutant'].apply(lambda x: all([(y[0] in AA_vocab) and (y[1:-1].isnumeric()) and (y[-1] in AA_vocab) for y in x.split(":")]))
# display(tmp_df[~tmp_df["mutant_valid_aa"]])

# tmp_df["positions"] = tmp_df[tmp_df["mutant_valid_aa"]]['mutant'].apply(lambda x: [int(y[1:-1]) for y in x.split(":")])
# unzip all the nested lists into a single list
# pos_list = tmp_df["positions"]
# positions_unique = set(flatten(tmp_df["positions"].dropna()))
# print(positions_unique)
# print(tmp_df[tmp_df["positions"].apply(lambda li: ~np.isnan(li) and 0 in li)])
# tmp_wt = "MLDDRARMEAAKKEKVEQILAEFQLQEEDLKKVMRRMQKEMDRGLRLETHEEASVKMLPTYVRSTPEGSEVGDFLSLDLGGTNFRVMLVKVGEGEEGQWSVKTKHQMYSIPEDAMTGTAEMLFDYISECISDFLDKHQMKHKKLPLGFTFSFPVRHEDIDKGILLNWTKGFKASGAEGNNVVGLLRDAIKRRGDFEMDVVAMVNDTVATMISCYYEDHQCEVGMIVGTGCNACYMEEMQNVELVEGDEGRMCVNTEWGAFGDSGELDEFLLEYDRLVDESSANPGQQLYEKLIGGKYMGELVRLVLLRLVDENLLFHGEASEQLRTRGAFETRFVSQVESDTGDRKQIYNILSTLGLRPSTTDCDIVRRACESVSTRAAHMCSAGLAGVINRMRESRSEDVMRITVGVDGSVYKLHPSFKERFHASVRRLTPSCEITFIESEEGSGRGAALVSAVACKKACMLGQ"
tmp_df_clean = DMS_file_cleanup(filename, target_seq=target_seq, DMS_binarization_threshold=dms_bin_threshold, DMS_mutant_column=row["raw_DMS_mutant_column"], DMS_phenotype_name=row["raw_DMS_phenotype_name"], DMS_directionality=row["raw_DMS_directionality"], debug=True, strict_wt_check=True)
def check_synon_multiple(mutant):
    muts = mutant.split(":")
    if len(muts) > 1:
        for mut in muts:
            if mut[0] == mut[-1]:
                return True 
    return False 
display(tmp_df_clean)
# tmp_df_clean.to_csv(f"/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/DMS_assays/substitutions/{dms_filename}",index=False)

Number of dropped mutant formats: 5 / 4039 = 0.1%
Number of dropped invalid amino acids: 5 / 4039 = 0.1%
Number of dropped synonymous mutants: 11 / 4039 = 0.3%
Number of dropped mutants not matching WT: 0 / 4039 = 0.0%
Dropped mutants: Length before: 4039, length after: 4023: 0.4% dropped
HERE
D28Q:T32Q
['D28Q', 'T32Q']
HERE
D28Q:T32E
['D28Q', 'T32E']
HERE
D28Q:T32N
['D28Q', 'T32N']
HERE
D28Q:T32H
['D28Q', 'T32H']
HERE
D28Q:T32D
['D28Q', 'T32D']
HERE
D28Q:T32R
['D28Q', 'T32R']
HERE
D28Q:T32K
['D28Q', 'T32K']
HERE
D28Q:T32T
['D28Q']
HERE
D28Q:T32S
['D28Q', 'T32S']
HERE
D28Q:T32A
['D28Q', 'T32A']
HERE
D28Q:T32G
['D28Q', 'T32G']
HERE
D28Q:T32M
['D28Q', 'T32M']
HERE
D28Q:T32L
['D28Q', 'T32L']
HERE
D28Q:T32V
['D28Q', 'T32V']
HERE
D28Q:T32I
['D28Q', 'T32I']
HERE
D28Q:T32W
['D28Q', 'T32W']
HERE
D28Q:T32Y
['D28Q', 'T32Y']
HERE
D28Q:T32F
['D28Q', 'T32F']
HERE
D28Q:T32P
['D28Q', 'T32P']
HERE
D28E:T32Q
['D28E', 'T32Q']
HERE
D28E:T32E
['D28E', 'T32E']
HERE
D28E:T32N
['D28E', 'T32N']
HERE
D28E:T32H

,mutant,mutated_sequence,DMS_score,DMS_score_bin
0,A10C,YFQGQLNAMCHQIQEMFPQVPYHLVLQDLQLTRSVEITTDNILEGRI,-1.299780,1
1,A10F,YFQGQLNAMFHQIQEMFPQVPYHLVLQDLQLTRSVEITTDNILEGRI,-1.066926,1
2,A10G,YFQGQLNAMGHQIQEMFPQVPYHLVLQDLQLTRSVEITTDNILEGRI,-1.843024,0
3,A10H,YFQGQLNAMHHQIQEMFPQVPYHLVLQDLQLTRSVEITTDNILEGRI,-2.299680,0
4,A10I,YFQGQLNAMIHQIQEMFPQVPYHLVLQDLQLTRSVEITTDNILEGRI,-0.387281,1
...,...,...,...,...
2967,Y22R,YFQGQLNAMAHQIQEMFPQVPRHLVLQDLQLTRSVEITTDNILEGRI,-0.747871,1
2968,Y22S,YFQGQLNAMAHQIQEMFPQVPSHLVLQDLQLTRSVEITTDNILEGRI,-1.266067,1
2969,Y22T,YFQGQLNAMAHQIQEMFPQVPTHLVLQDLQLTRSVEITTDNILEGRI,-1.103759,1
2970,Y22V,YFQGQLNAMAHQIQEMFPQVPVHLVLQDLQLTRSVEITTDNILEGRI,-0.908841,1


In [11]:
overwrite = True
overwrite = input("Are you sure you want to overwrite the directory? (y/n)").upper() == "Y"

Are you sure you want to overwrite the directory? (y/n) Y


In [13]:
# Process each DMS and drop in folder
# TODO skip_existing of same length file?
# TODO some files with multiple phenotypes might just be DMS_id.csv, should be able to use them too ideally
# assert overwrite, "Not overwriting"
keep_versions = ["1"]
processed_list = os.listdir(dms_processed_folder_out)
# to_reprocess = """
# F7YBW7_MESOW_Ding_2023
# AMFR_HUMAN_Rocklin_2023_4G3O
# BBC1_YEAST_Rocklin_2023_1TG0
# BCHB_CHLTE_Rocklin_2023_2KRU
# CATR_CHLRE_Rocklin_2023_2AMI
# CBPA2_HUMAN_Rocklin_2023_1O6X
# CBX4_HUMAN_Rocklin_2023_2K28
# CSN4_MOUSE_Rocklin_2023_1UFM
# CUE1_YEAST_Rocklin_2023_2MYX
# DNJA1_HUMAN_Rocklin_2023_2LO1
# DOCK1_MOUSE_Rocklin_2023_2M0Y
# EPHB2_HUMAN_Rocklin_2023_1F0M
# FECA_ECOLI_Rocklin_2023_2D1U
# HECD1_HUMAN_Rocklin_2023_3DKM
# ISDH_STAAW_Rocklin_2023_2LHR
# MAFG_MOUSE_Rocklin_2023_1K1V
# MBD11_ARATH_Rocklin_2023_6ACV
# MYO3_YEAST_Rocklin_2023_2BTT
# NKX31_HUMAN_Rocklin_2023_2L9R
# NUSA_ECOLI_Rocklin_2023_1WCL
# NUSG_MYCTU_Rocklin_2023_2MI6
# OBSCN_HUMAN_Rocklin_2023_1V1C
# ODP2_GEOSE_Rocklin_2023_1W4G
# PIN1_HUMAN_Rocklin_2023_1I6C
# PITX2_HUMAN_Rocklin_2023_2L7M
# POLG_PESV_Rocklin_2023_2MXD
# PR40A_HUMAN_Rocklin_2023_1UZC
# PSAE_SYNP2_Rocklin_2023_1PSE
# RAD_ANTMA_Rocklin_2023_2CJJ
# RBP1_HUMAN_Rocklin_2023_2KWH
# RCD1_ARATH_Rocklin_2023_5OAO
# RCRO_LAMBD_Rocklin_2023_1ORC
# RD23A_HUMAN_Rocklin_2023_1IFY
# RFAH_ECOLI_Rocklin_2023_2LCL
# RL20_AQUAE_Rocklin_2023_1GYZ
# RPC1_BP434_Rocklin_2023_1R69
# SAV1_MOUSE_Rocklin_2023_2YSB
# SDA_BACSU_Rocklin_2023_1PV0
# SPA_STAAU_Rocklin_2023_1LP1
# SPG2_STRSG_Rocklin_2023_5UBS
# SPTN1_CHICK_Rocklin_2023_1TUD
# SR43C_ARATH_Rocklin_2023_2N88
# SRBS1_HUMAN_Rocklin_2023_2O2W
# TCRG1_MOUSE_Rocklin_2023_1E0L
# THO1_YEAST_Rocklin_2023_2WQG
# TNKS2_HUMAN_Rocklin_2023_5JRT
# UBE4B_HUMAN_Rocklin_2023_3L1X
# UBR5_HUMAN_Rocklin_2023_1I2T
# VILI_CHICK_Rocklin_2023_1YU5
# YAIA_ECOLI_Rocklin_2023_2KVT
# YNZC_BACSU_Rocklin_2023_2JVD
# """.split("\n")
pbar = tqdm(ref_df[ref_df["ProteinGym_version"] == 1].dropna(subset=necessary_cols, how='any').iterrows())
# pbar = tqdm(ref_df[ref_df["DMS_id"].isin(to_reprocess)].dropna(subset=necessary_cols,how='any').iterrows())
for i, row in pbar:
    pbar.set_postfix_str(row["DMS_filename"])
    if row["DMS_filename"] in processed_list and not overwrite:
        print(f"{row['DMS_filename']} is already processed. Skipping")
        continue 
    print(row["DMS_filename"])
    # This row was removed
    if row["DMS_id"] in ["HXK4_HUMAN_Gersing_2023_activity"]:
        print(f"Skipping {row['DMS_id']}")
        continue
    filename = dms_preprocessed_folder + "/" + row["DMS_filename"]
    if not os.path.isfile(filename):
        print("Missing file: "+filename)
        continue
    dest = dms_processed_folder_out + "/" + row["DMS_filename"]
    if os.path.isfile(dest) and not overwrite:
        print(f"File already exists: {dest}")
        continue
    tmp_df = pd.read_csv(filename)
    assert not len(tmp_df) == 0

    # Ignoring start and end seq
    tmp_df_clean = DMS_file_cleanup(filename, target_seq=row["target_seq"], DMS_binarization_threshold=row["DMS_binarization_cutoff"], DMS_mutant_column=row["raw_DMS_mutant_column"], DMS_phenotype_name=row["raw_DMS_phenotype_name"], DMS_directionality=row["raw_DMS_directionality"], debug=True, strict_wt_check=True)
    display(tmp_df_clean)
    # Write out to processed folder
    tmp_df_clean.to_csv(dest, index=False)

print("Done")

0it [00:00, ?it/s]

F7YBW7_MESOW_Ding_2023.csv
Number of dropped mutant formats: 0 / 10656 = 0.0%
Number of dropped invalid amino acids: 2620 / 10656 = 24.6%
Number of dropped synonymous mutants: 16 / 10656 = 0.2%
Number of dropped mutants not matching WT: 0 / 10656 = 0.0%
Dropped mutants: Length before: 10656, length after: 8020: 24.7% dropped
Debug: Number of duplicate mutants: 98 (1.22%)
Final length: 7922


,mutant,mutated_sequence,DMS_score,DMS_score_bin
0,A81D,MANVEKMSVAVTPQQAAVMREAVEAGEYATASEIVREAVRDWLAKR...,0.883542,1
1,A81D:R82G,MANVEKMSVAVTPQQAAVMREAVEAGEYATASEIVREAVRDWLAKR...,0.107614,0
2,A81F:R82D,MANVEKMSVAVTPQQAAVMREAVEAGEYATASEIVREAVRDWLAKR...,0.715109,1
3,A81G,MANVEKMSVAVTPQQAAVMREAVEAGEYATASEIVREAVRDWLAKR...,0.932408,1
4,A81K:R82K,MANVEKMSVAVTPQQAAVMREAVEAGEYATASEIVREAVRDWLAKR...,0.266513,0
...,...,...,...,...
7917,R82G,MANVEKMSVAVTPQQAAVMREAVEAGEYATASEIVREAVRDWLAKR...,0.954072,1
7918,R82L,MANVEKMSVAVTPQQAAVMREAVEAGEYATASEIVREAVRDWLAKR...,0.980336,1
7919,R82P,MANVEKMSVAVTPQQAAVMREAVEAGEYATASEIVREAVRDWLAKR...,0.868006,1
7920,R82Q,MANVEKMSVAVTPQQAAVMREAVEAGEYATASEIVREAVRDWLAKR...,1.027203,1


AMFR_HUMAN_Rocklin_2023_4G3O.csv
Number of dropped mutant formats: 5 / 4039 = 0.1%
Number of dropped invalid amino acids: 5 / 4039 = 0.1%
Number of dropped synonymous mutants: 11 / 4039 = 0.3%
Number of dropped mutants not matching WT: 0 / 4039 = 0.0%
Dropped mutants: Length before: 4039, length after: 4023: 0.4% dropped
Debug: Number of duplicate mutants: 1051 (26.12%)
Final length: 2972


,mutant,mutated_sequence,DMS_score,DMS_score_bin
0,A10C,YFQGQLNAMCHQIQEMFPQVPYHLVLQDLQLTRSVEITTDNILEGRI,-1.299780,1
1,A10F,YFQGQLNAMFHQIQEMFPQVPYHLVLQDLQLTRSVEITTDNILEGRI,-1.066926,1
2,A10G,YFQGQLNAMGHQIQEMFPQVPYHLVLQDLQLTRSVEITTDNILEGRI,-1.843024,0
3,A10H,YFQGQLNAMHHQIQEMFPQVPYHLVLQDLQLTRSVEITTDNILEGRI,-2.299680,0
4,A10I,YFQGQLNAMIHQIQEMFPQVPYHLVLQDLQLTRSVEITTDNILEGRI,-0.387281,1
...,...,...,...,...
2967,Y22R,YFQGQLNAMAHQIQEMFPQVPRHLVLQDLQLTRSVEITTDNILEGRI,-0.747871,1
2968,Y22S,YFQGQLNAMAHQIQEMFPQVPSHLVLQDLQLTRSVEITTDNILEGRI,-1.266067,1
2969,Y22T,YFQGQLNAMAHQIQEMFPQVPTHLVLQDLQLTRSVEITTDNILEGRI,-1.103759,1
2970,Y22V,YFQGQLNAMAHQIQEMFPQVPVHLVLQDLQLTRSVEITTDNILEGRI,-0.908841,1


...

YAIA_ECOLI_Rocklin_2023_2KVT.csv
Number of dropped mutant formats: 5 / 2018 = 0.2%
Number of dropped invalid amino acids: 5 / 2018 = 0.2%
Number of dropped synonymous mutants: 4 / 2018 = 0.2%
Number of dropped mutants not matching WT: 0 / 2018 = 0.0%
Dropped mutants: Length before: 2018, length after: 2009: 0.4% dropped
Debug: Number of duplicate mutants: 119 (5.92%)
Final length: 1890


,mutant,mutated_sequence,DMS_score,DMS_score_bin
0,A25C,PREAYIVTIEKGKPGQTVTWYQLRCDHPKPDSLISEHPTAQEAMDA...,-0.071576,1
1,A25D,PREAYIVTIEKGKPGQTVTWYQLRDDHPKPDSLISEHPTAQEAMDA...,-2.026212,0
2,A25E,PREAYIVTIEKGKPGQTVTWYQLREDHPKPDSLISEHPTAQEAMDA...,-2.098785,0
3,A25F,PREAYIVTIEKGKPGQTVTWYQLRFDHPKPDSLISEHPTAQEAMDA...,-1.759015,1
4,A25G,PREAYIVTIEKGKPGQTVTWYQLRGDHPKPDSLISEHPTAQEAMDA...,-0.869425,1
...,...,...,...,...
1885,Y5R,PREARIVTIEKGKPGQTVTWYQLRADHPKPDSLISEHPTAQEAMDA...,-0.076517,1
1886,Y5S,PREASIVTIEKGKPGQTVTWYQLRADHPKPDSLISEHPTAQEAMDA...,-2.002802,0
1887,Y5T,PREATIVTIEKGKPGQTVTWYQLRADHPKPDSLISEHPTAQEAMDA...,-1.653230,1
1888,Y5V,PREAVIVTIEKGKPGQTVTWYQLRADHPKPDSLISEHPTAQEAMDA...,-1.562712,1


YNZC_BACSU_Rocklin_2023_2JVD.csv
Number of dropped mutant formats: 5 / 2479 = 0.2%
Number of dropped invalid amino acids: 5 / 2479 = 0.2%
Number of dropped synonymous mutants: 5 / 2479 = 0.2%
Number of dropped mutants not matching WT: 0 / 2479 = 0.0%
Dropped mutants: Length before: 2479, length after: 2469: 0.4% dropped
Debug: Number of duplicate mutants: 169 (6.84%)
Final length: 2300


,mutant,mutated_sequence,DMS_score,DMS_score_bin
0,A14D,MISNAKIARINELDAKAKAGVITEEEKAEQQKLRQEYLK,-0.044412,1
1,A14E,MISNAKIARINELEAKAKAGVITEEEKAEQQKLRQEYLK,0.069826,1
2,A14F,MISNAKIARINELFAKAKAGVITEEEKAEQQKLRQEYLK,-0.033247,1
3,A14G,MISNAKIARINELGAKAKAGVITEEEKAEQQKLRQEYLK,-0.756903,1
4,A14H,MISNAKIARINELHAKAKAGVITEEEKAEQQKLRQEYLK,-0.441117,1
...,...,...,...,...
2295,Y37R,MISNAKIARINELAAKAKAGVITEEEKAEQQKLRQERLK,-0.453744,1
2296,Y37S,MISNAKIARINELAAKAKAGVITEEEKAEQQKLRQESLK,-0.602686,1
2297,Y37T,MISNAKIARINELAAKAKAGVITEEEKAEQQKLRQETLK,-0.570633,1
2298,Y37V,MISNAKIARINELAAKAKAGVITEEEKAEQQKLRQEVLK,-0.283027,1


Done


## Quickly compare singles counts with reference file

In [27]:
keep_versions = ["1"]
processed_list = os.listdir(dms_processed_folder_out)
filenames = []
single_lengths = []
multiple_lengths = []
pbar = tqdm(ref_df[ref_df["ProteinGym_version"] == 1].dropna(subset=necessary_cols, how='any').iterrows())
for i, row in pbar:
    pbar.set_postfix_str(row["DMS_filename"])
    assert row["DMS_filename"] in processed_list, f"{row['DMS_filename']} is not in {dms_processed_folder_out}"
    df = pd.read_csv(dms_processed_folder_out + "/" + row["DMS_filename"])
    num_multiple = df["mutant"].str.contains(":").sum()
    num_single = len(df["mutant"]) - num_multiple  # wt has been filtered out by this point
    filenames.append(row["DMS_filename"])
    single_lengths.append(num_single)
    multiple_lengths.append(num_multiple)
    

print("Done")

0it [00:00, ?it/s]

Done


In [28]:
df_lens = pd.DataFrame({"DMS_filename": filenames, "DMS_number_single_mutants": single_lengths, "DMS_number_multiple_mutants": multiple_lengths})
df_lens.head()

,DMS_filename,DMS_number_single_mutants,DMS_number_multiple_mutants
0,PAI1_HUMAN_Huttinger_2021.csv,5345,0
1,HXK4_HUMAN_Gersing_2023_abundance.csv,8396,0
2,PHOT_CHLRE_Chen_2023_singles.csv,2121,0
3,PHOT_CHLRE_Chen_2023_multiples.csv,20,165407
4,S22A1_HUMAN_Yee_2023_abundance.csv,9803,0


In [35]:
# There are a lot of differences here - interesting 
df_merged = ref_df.merge(df_lens, on="DMS_filename", how="outer")
def gentle_int_float(i):
    try :
        return float(i)
    except ValueError:
        # e.g. "6,289"
        return int(i.split(",")[0]) * 1000.0 + float(i.split(",")[1])
df_merged["DMS_number_single_mutants_x"] = df_merged["DMS_number_single_mutants_x"].apply(gentle_int_float)
# Singles
df_merged[(df_merged["ProteinGym_version"] == 1) & (df_merged["DMS_number_single_mutants_x"].astype(float) != df_merged["DMS_number_single_mutants_y"])][["DMS_filename", "DMS_number_single_mutants_x", "DMS_number_single_mutants_y"]]

,DMS_filename,DMS_number_single_mutants_x,DMS_number_single_mutants_y
95,PAI1_HUMAN_Huttinger_2021.csv,1963.0,5345.0
96,HXK4_HUMAN_Gersing_2023_abundance.csv,8395.0,8396.0
97,PHOT_CHLRE_Chen_2023_singles.csv,2184.0,2121.0
98,PHOT_CHLRE_Chen_2023_multiples.csv,0.0,20.0
99,S22A1_HUMAN_Yee_2023_abundance.csv,11029.0,9803.0
...,...,...,...
210,OPSD_HUMAN_Wan_2019.csv,168.0,165.0
211,CAR11_HUMAN_Meitlis_2020_lof.csv,2630.0,2395.0
212,CAR11_HUMAN_Meitlis_2020_gof.csv,NaN,2374.0
214,GLPA_HUMAN_Elazar_2016.csv,285.0,245.0


## Process held assays separately

In [19]:
# Process only the Nathan ones on their own
failures_skip_manually = []
count = 0
pbar = tqdm(ref_df[ref_df["reviewed_by"] == "Nathan"].iterrows())
overwrite = True
if overwrite:
    print("Overwriting files")
for i, row in pbar:
    pbar.set_postfix_str(row["DMS_filename"])
    print(row["DMS_filename"])
    filename = dms_preprocessed_folder_nathan + "/" + row["DMS_filename"]
    # if "escape" in filename:
    #     print("tmp: Skipping escape assays")
    #     continue
    if row["DMS_filename"] in failures_skip_manually:
        print(f"Skipping {row['DMS_filename']} manually - tmp")
        continue
    if not os.path.isfile(filename):
        print("Missing file: "+filename)
        continue
    tmp_df = pd.read_csv(filename)
    assert not len(tmp_df) == 0

    # Check phenotype column in DMS
    # Ignoring start and end seq
    if pd.isna(row["raw_DMS_phenotype_name"]):
        print("No phenotype given, will use score")
        score_col = "score"
    else:
        score_col = row["raw_DMS_phenotype_name"]
    tmp_df_clean = DMS_file_cleanup(filename, target_seq=row["target_seq"], DMS_mutant_column="mut_proteingym", DMS_binarization_threshold=None, DMS_phenotype_name=score_col, DMS_directionality=row["raw_DMS_directionality"], debug=True, strict_wt_check=True)
    
    count += 1
    # Write out to processed folder
    if overwrite:
        tmp_df_clean.to_csv(dms_processed_folder_out + "/" + row["DMS_filename"], index=False)

print("Done")
print(count)

0it [00:00, ?it/s]

Overwriting files
HSP82_YEAST_Hietpas_2011.csv
Number of dropped mutant formats: 0 / 518 = 0.0%
Number of dropped invalid amino acids: 0 / 518 = 0.0%
Number of dropped synonymous mutants: 0 / 518 = 0.0%
Number of dropped mutants not matching WT: 0 / 518 = 0.0%
Debug: Number of duplicate mutants: 347 (66.99%)
No binarization threshold given, will outputs NaNs in that column for now.
Final length: 171
I6TAH8_I68A0_Bloom_2014.csv
Number of dropped mutant formats: 0 / 9462 = 0.0%
Number of dropped invalid amino acids: 0 / 9462 = 0.0%
Number of dropped synonymous mutants: 0 / 9462 = 0.0%
Number of dropped mutants not matching WT: 0 / 9462 = 0.0%
Debug: Number of duplicate mutants: 0 (0.00%)
No binarization threshold given, will outputs NaNs in that column for now.
Final length: 9462
ARRS_BOVIN_Ostermaier_2014.csv
Number of dropped mutant formats: 1 / 404 = 0.2%
Number of dropped invalid amino acids: 1 / 404 = 0.2%
Number of dropped synonymous mutants: 0 / 404 = 0.0%
Number of dropped mutant

In [79]:
# Tmp get indices of Nathan assays
score_indices = []
for i, row in ref_df[ref_df["reviewed_by"] == "Nathan"].iterrows():
    if os.path.isfile(dms_processed_folder_out + "/" + row["DMS_filename"]):
        print(row["DMS_filename"])
        score_indices.append(i)

def indices_runs(indices):
    # Print consecutive runs together as a range, for sbatch
    runs = []
    for i in indices:
        if i == indices[0]:
            start = i
            current = i
        elif i == current + 1:
            current += 1
        else:
            if start == current:
                runs.append(str(start))
            else:
                runs.append(f"{start}-{current}")
            start = i
            current = i
    return ",".join(runs)
print(score_indices)
indices_runs(score_indices)

HSP82_YEAST_Hietpas_2011.csv
I6TAH8_I68A0_Bloom_2014.csv
ARRS_BOVIN_Ostermaier_2014.csv
SOX17_MOUSE_Veerapandian_2018.csv
SOX2_MOUSE_Veerapandian_2018.csv
A4_HUMAN_Gray_2019.csv
M1_I000X_Hom_2019.csv
Q53Z42_HUMAN_McShan_2019_expression.csv
Q53Z42_HUMAN_McShan_2019_binding-TAPBPR.csv
A0A3G4RHW3_ATRBE_Wrenbeck_2019.csv
B3VI55_LIPST_Wrenbeck_2019.csv
RPC1_LAMBD_Li_2019_low-expression.csv
RPC1_LAMBD_Li_2019_high-expression.csv
RASK_HUMAN_Ursu_2020.csv
P53_HUMAN_Ursu_2020.csv
SPIKE_SARS2_Greany_2020_escape-COV2-2050.csv
SPIKE_SARS2_Greany_2020_escape-COV2-2082.csv
SPIKE_SARS2_Greany_2020_escape-COV2-2094.csv
SPIKE_SARS2_Greany_2020_escape-COV2-2096.csv
SPIKE_SARS2_Greany_2020_escape-COV2-2165.csv
SPIKE_SARS2_Greany_2020_escape-COV2-2479.csv
SPIKE_SARS2_Greany_2020_escape-COV2-2499.csv
SPIKE_SARS2_Greany_2020_escape-COV2-2677.csv
SPIKE_SARS2_Greany_2020_escape-COV2-2832.csv
SPIKE_SARS2_Greany_2020_escape-rCR3022.csv
B2L11_HUMAN_Dutta_2010_binding-Mcl-1.csv
B2L11_HUMAN_Dutta_2010_binding-Bcl-

'124-126,132-155,164-166'

## Reprocess Flynn from v0.1 with new 37C phenotype

In [21]:
dms_id = "HSP82_YEAST_Flynn_2019"
row = ref_df[ref_df["DMS_id"] == dms_id]
assert len(row) == 1
row = row.iloc[0]
filename = dms_preprocessed_folder + row["DMS_filename"]

tmp_df = pd.read_csv(filename)
assert not len(tmp_df) == 0

# Check phenotype column in DMS
# Ignoring start and end seq
if pd.isna(row["raw_DMS_phenotype_name"]):
    print("No phenotype given, will use score")
    score_col = "score"
else:
    score_col = row["raw_DMS_phenotype_name"]

assert score_col == "s (37°C)"
tmp_df_clean = DMS_file_cleanup(filename, target_seq=row["target_seq"], DMS_mutant_column=row["raw_DMS_mutant_column"], DMS_binarization_threshold=row["DMS_binarization_cutoff"], DMS_phenotype_name=score_col, DMS_directionality=row["raw_DMS_directionality"], debug=True, strict_wt_check=True)

dest = dms_processed_folder_out + "/" + row["DMS_filename"]
assert not os.path.isfile(dest)
tmp_df_clean.to_csv(dest, index=False)

Number of dropped mutant formats: 0 / 14868 = 0.0%
Number of dropped invalid amino acids: 708 / 14868 = 4.8%
Number of dropped synonymous mutants: 708 / 14868 = 4.8%
Number of dropped mutants not matching WT: 0 / 14868 = 0.0%
Dropped mutants: Length before: 14868, length after: 13452: 9.5% dropped
Debug: Number of duplicate mutants: 0 (0.00%)
Final length: 13294
